# 03. 인과추론 심화 — Propensity Score Matching, CATE, Uplift Modeling

## 목표
단순 A/B 비교를 넘어, **인과적 추천 효과**를 추정한다.

### 분석 항목
1. **Propensity Score Matching (PSM)** — 공변량 균형을 맞춘 인과 효과 추정
2. **CATE (Conditional Average Treatment Effect)** — "어떤 유저에게 추천이 효과적인가"
3. **Uplift Modeling** — 유저를 4분류 (Persuadables / Sure Things / Lost Causes / Sleeping Dogs)

---

In [ ]:
import sys
sys.path.append('../src')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from pathlib import Path
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors
from sklearn.ensemble import GradientBoostingClassifier, GradientBoostingRegressor
from sklearn.model_selection import cross_val_predict

plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12
sns.set_style('whitegrid')

DATA_DIR = Path('../data')

# 데이터 로드
log_rand = pd.read_csv(DATA_DIR / 'kuairand/KuaiRand-Pure/data/log_random_4_22_to_5_08_pure.csv')
log_std = pd.read_csv(DATA_DIR / 'kuairand/KuaiRand-Pure/data/log_standard_4_22_to_5_08_pure.csv')
user_feat = pd.read_csv(DATA_DIR / 'kuairand/KuaiRand-Pure/data/user_features_pure.csv')
video_stat = pd.read_csv(DATA_DIR / 'kuairand/KuaiRand-Pure/data/video_features_statistic_pure.csv')

control = log_rand[log_rand['is_rand'] == 1].copy()
treatment = log_std[log_std['is_rand'] == 0].copy()

print(f"Control: {len(control):,} | Treatment: {len(treatment):,}")
print("Setup complete!")

## 1. 유저 수준 데이터 집계

노출 단위가 아닌 **유저 단위**로 집계하여 인과추론 수행. 각 유저의 랜덤/추천 노출에 대한 평균 반응을 계산한다.

In [ ]:
# 유저별 평균 반응 집계
feedback_cols = ['is_click', 'is_like', 'is_follow', 'is_comment', 'long_view']

ctrl_user = control.groupby('user_id').agg(
    **{f'ctrl_{c}': (c, 'mean') for c in feedback_cols},
    ctrl_play_time=('play_time_ms', 'mean'),
    ctrl_count=('is_click', 'count')
).reset_index()

treat_user = treatment.groupby('user_id').agg(
    **{f'treat_{c}': (c, 'mean') for c in feedback_cols},
    treat_play_time=('play_time_ms', 'mean'),
    treat_count=('is_click', 'count')
).reset_index()

# 두 그룹 모두에 존재하는 유저만 (within-subject)
both_users = set(ctrl_user['user_id']) & set(treat_user['user_id'])
print(f"양쪽 모두 존재하는 유저: {len(both_users):,}")

# 유저 특성 조인
# 숫자형만 사용
user_num_cols = ['user_id', 'is_lowactive_period', 'is_live_streamer', 'is_video_author',
                 'follow_user_num', 'fans_user_num', 'friend_user_num', 'register_days']
user_feat_num = user_feat[user_num_cols].copy()
# -124 등 결측 표기 → NaN 처리
for c in user_num_cols[1:]:
    user_feat_num[c] = user_feat_num[c].apply(lambda x: np.nan if x < 0 else x)

user_feat_num = user_feat_num.dropna()
print(f"유효 유저 특성: {len(user_feat_num):,}")

# 활동도 추가 (원핫)
active_map = {'full_active': 4, 'high_active': 3, 'middle_active': 2, 'low_active': 1, 'UNKNOWN': 0}
user_feat_num['active_level'] = user_feat['user_active_degree'].map(active_map).fillna(0)

user_feat_num.head()

## 2. Propensity Score Matching (PSM)

노출 단위를 샘플링하여 Treatment/Control 데이터프레임을 구성하고, 유저 특성 기반으로 Propensity Score를 추정한 뒤 매칭한다.

In [ ]:
# 성능을 위해 각 그룹에서 50K 샘플링
np.random.seed(42)
n_sample = 50_000

ctrl_sample = control.sample(n=n_sample, random_state=42)
treat_sample = treatment.sample(n=min(n_sample, len(treatment)), random_state=42)

# Treatment 플래그 추가
ctrl_sample['treated'] = 0
treat_sample['treated'] = 1
combined = pd.concat([ctrl_sample, treat_sample], ignore_index=True)

# 유저 특성 조인
combined = combined.merge(user_feat_num, on='user_id', how='inner')
print(f"Combined dataset: {len(combined):,} rows")

# 공변량 (Covariates)
covariates = ['is_lowactive_period', 'is_live_streamer', 'is_video_author',
              'follow_user_num', 'fans_user_num', 'friend_user_num', 'register_days', 'active_level']

X = combined[covariates].values
y = combined['treated'].values

# Propensity Score 추정 (Logistic Regression)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

ps_model = LogisticRegression(max_iter=1000, random_state=42)
ps_model.fit(X_scaled, y)
combined['propensity_score'] = ps_model.predict_proba(X_scaled)[:, 1]

print(f"\nPropensity Score 분포:")
print(combined.groupby('treated')['propensity_score'].describe().round(4))

In [ ]:
# Propensity Score 분포 시각화
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# PS 히스토그램
axes[0].hist(combined[combined['treated']==0]['propensity_score'], bins=50, alpha=0.6, 
             label='Control', color='#3498db', density=True)
axes[0].hist(combined[combined['treated']==1]['propensity_score'], bins=50, alpha=0.6, 
             label='Treatment', color='#e74c3c', density=True)
axes[0].set_title('Propensity Score Distribution')
axes[0].set_xlabel('Propensity Score')
axes[0].legend()

# Common Support 확인
axes[1].hist(combined[combined['treated']==0]['propensity_score'], bins=50, alpha=0.6, 
             label='Control', color='#3498db', density=True, cumulative=True)
axes[1].hist(combined[combined['treated']==1]['propensity_score'], bins=50, alpha=0.6, 
             label='Treatment', color='#e74c3c', density=True, cumulative=True)
axes[1].set_title('Cumulative PS Distribution (Common Support Check)')
axes[1].set_xlabel('Propensity Score')
axes[1].legend()

plt.tight_layout()
plt.savefig('../notebooks/figures/03_propensity_score.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Nearest-Neighbor Matching on Propensity Score
treated_df = combined[combined['treated'] == 1].copy()
control_df = combined[combined['treated'] == 0].copy()

# 1:1 매칭 (caliper = 0.05)
nn = NearestNeighbors(n_neighbors=1, metric='euclidean')
nn.fit(control_df[['propensity_score']].values)

distances, indices = nn.kneighbors(treated_df[['propensity_score']].values)

# Caliper 적용
caliper = 0.05
mask = distances.flatten() < caliper
matched_treat = treated_df[mask].copy()
matched_ctrl_idx = indices.flatten()[mask]
matched_ctrl = control_df.iloc[matched_ctrl_idx].copy()

print(f"매칭 전 — Treatment: {len(treated_df):,}, Control: {len(control_df):,}")
print(f"매칭 후 — Treatment: {len(matched_treat):,}, Control: {len(matched_ctrl):,}")
print(f"Caliper ({caliper}) 내 매칭률: {mask.mean()*100:.1f}%")

# 매칭 후 공변량 균형 확인 (Standardized Mean Difference)
smd_before = []
smd_after = []
for cov in covariates:
    # Before matching
    d_before = (treated_df[cov].mean() - control_df[cov].mean()) / np.sqrt(
        (treated_df[cov].var() + control_df[cov].var()) / 2)
    # After matching
    d_after = (matched_treat[cov].mean() - matched_ctrl[cov].mean()) / np.sqrt(
        (matched_treat[cov].var() + matched_ctrl[cov].var()) / 2)
    smd_before.append(abs(d_before))
    smd_after.append(abs(d_after))

balance_df = pd.DataFrame({
    'Covariate': covariates,
    'SMD Before': smd_before,
    'SMD After': smd_after,
    'Improved': ['✅' if a < b else '❌' for a, b in zip(smd_after, smd_before)]
})
print("\n=== Covariate Balance (|SMD|) ===")
balance_df

In [ ]:
# Love Plot — 매칭 전후 공변량 균형 시각화
fig, ax = plt.subplots(figsize=(10, 6))

y_pos = np.arange(len(covariates))
ax.scatter(smd_before, y_pos, marker='o', s=100, color='#e74c3c', label='Before Matching', zorder=3)
ax.scatter(smd_after, y_pos, marker='s', s=100, color='#2ecc71', label='After Matching', zorder=3)

for i in range(len(covariates)):
    ax.plot([smd_before[i], smd_after[i]], [y_pos[i], y_pos[i]], 'k-', alpha=0.3)

ax.axvline(x=0.1, color='gray', linestyle='--', alpha=0.5, label='|SMD| = 0.1 threshold')
ax.set_yticks(y_pos)
ax.set_yticklabels(covariates)
ax.set_xlabel('|Standardized Mean Difference|')
ax.set_title('Love Plot — Covariate Balance Before/After PSM')
ax.legend()

plt.tight_layout()
plt.savefig('../notebooks/figures/03_love_plot.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ATT (Average Treatment Effect on the Treated) — 매칭 후
print("=== ATT after PSM (매칭 후 인과 효과 추정) ===\n")

att_results = []
for metric in ['is_click', 'is_like', 'is_follow', 'is_comment', 'long_view']:
    treat_mean = matched_treat[metric].mean()
    ctrl_mean = matched_ctrl[metric].mean()
    att = treat_mean - ctrl_mean
    
    # 단순 A/B 차이 (매칭 전)
    naive_diff = treated_df[metric].mean() - control_df[metric].mean()
    
    att_results.append({
        'Metric': metric,
        'Naive Diff': f"{naive_diff:.4f}",
        'ATT (PSM)': f"{att:.4f}",
        'Bias Reduction': f"{(1 - abs(att)/abs(naive_diff))*100:.1f}%" if naive_diff != 0 else 'N/A'
    })

att_df = pd.DataFrame(att_results)
print("Naive Diff = 단순 평균 차이 (편향 포함)")
print("ATT (PSM) = 매칭 후 인과 효과 추정치\n")
att_df

## 3. CATE — 이질적 처치 효과 (Heterogeneous Treatment Effects)

"어떤 유저에게 추천이 더 효과적인가?"를 분석한다. DR-Learner 접근법 사용.

DR-Learner는 T-Learner의 모델 추정값을 실제 관측값으로 보정하는 이중강건(Doubly Robust) 방식이다:
- **Step 1**: GBM 2개로 μ₁(x), μ₀(x) 추정 (T-Learner와 동일)
- **Step 2**: DR 의사결과(pseudo-outcome) 계산 — 실제 관측값으로 모델 오차 보정
- **Step 3**: DR 의사결과를 타겟으로 최종 CATE 모델 학습

In [ ]:
# DR-Learner: T-Learner 기반 + 실제 관측값으로 DR 보정
# μ₁(x) = E[Y|X=x, T=1], μ₀(x) = E[Y|X=x, T=0]
# DR pseudo-outcome: ψ = μ₁-μ₀ + T*(Y-μ₁)/e(x) - (1-T)*(Y-μ₀)/(1-e(x))
# CATE(x) = E[ψ | X=x]  ← 최종 GBM으로 스무딩

target = 'is_click'

X_treat = combined[combined['treated'] == 1][covariates].values
y_treat = combined[combined['treated'] == 1][target].values
X_ctrl = combined[combined['treated'] == 0][covariates].values
y_ctrl = combined[combined['treated'] == 0][target].values

# Step 1: T-Learner와 동일하게 μ₁, μ₀ 추정
model_treat = GradientBoostingClassifier(n_estimators=100, max_depth=4, random_state=42)
model_ctrl  = GradientBoostingClassifier(n_estimators=100, max_depth=4, random_state=42)

model_treat.fit(X_treat, y_treat)
model_ctrl.fit(X_ctrl, y_ctrl)

X_all = combined[covariates].values
mu1 = model_treat.predict_proba(X_all)[:, 1]  # 추천 시 예상 CTR
mu0 = model_ctrl.predict_proba(X_all)[:, 1]   # 랜덤 시 예상 CTR

# Step 2: DR 의사결과 계산
T = combined['treated'].values
Y = combined[target].values
ps = combined['propensity_score'].values
ps_clip = np.clip(ps, 0.05, 0.95)  # 극단값 클리핑 (분산 폭발 방지)

dr_pseudo = (mu1 - mu0
             + T * (Y - mu1) / ps_clip
             - (1 - T) * (Y - mu0) / (1 - ps_clip))

# Step 3: DR 의사결과로 최종 CATE 모델 학습 (스무딩)
dr_model = GradientBoostingRegressor(n_estimators=100, max_depth=4, random_state=42)
dr_model.fit(X_all, dr_pseudo)
combined['cate'] = dr_model.predict(X_all)

print(f"CATE 분포 (DR-Learner):")
print(f"  Mean:   {combined['cate'].mean():.4f}")
print(f"  Std:    {combined['cate'].std():.4f}")
print(f"  Min:    {combined['cate'].min():.4f}")
print(f"  Max:    {combined['cate'].max():.4f}")
print(f"  >0 비율: {(combined['cate'] > 0).mean()*100:.1f}%")

# T-Learner 단순 차이와 비교
t_learner_mean = (mu1 - mu0).mean()
print(f"\nT-Learner 평균 CATE: {t_learner_mean:.4f}")
print(f"DR-Learner 평균 CATE: {combined['cate'].mean():.4f}")
print(f"보정 차이: {combined['cate'].mean() - t_learner_mean:+.4f}")

In [ ]:
# CATE 분포 시각화
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# CATE 히스토그램
axes[0].hist(combined['cate'], bins=50, color='#9b59b6', alpha=0.7, edgecolor='white')
axes[0].axvline(x=0, color='black', linestyle='--', alpha=0.5)
axes[0].axvline(x=combined['cate'].mean(), color='red', linewidth=2, label=f"Mean CATE: {combined['cate'].mean():.4f}")
axes[0].set_title('CATE Distribution (DR-Learner)')
axes[0].set_xlabel('CATE (Recommendation Effect on CTR)')
axes[0].set_ylabel('Count')
axes[0].legend()

# 활동 수준별 CATE
cate_by_active = combined.groupby('active_level')['cate'].mean().sort_index()
active_labels = {0: 'Unknown', 1: 'Low', 2: 'Middle', 3: 'High', 4: 'Full'}
x_labels = [active_labels.get(k, str(k)) for k in cate_by_active.index]
colors = ['#2ecc71' if v > 0 else '#e74c3c' for v in cate_by_active.values]

axes[1].bar(x_labels, cate_by_active.values, color=colors, alpha=0.8)
axes[1].axhline(y=0, color='black', linewidth=0.5)
axes[1].set_title('Average CATE by User Activity Level')
axes[1].set_xlabel('Activity Level')
axes[1].set_ylabel('CATE (CTR Lift from Recommendation)')

for j, (label, val) in enumerate(zip(x_labels, cate_by_active.values)):
    axes[1].text(j, val + 0.002, f'{val:.4f}', ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig('../notebooks/figures/03_cate_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Uplift Modeling — 유저 4분류

CATE를 기반으로 유저를 4가지 유형으로 분류한다:
- **Persuadables**: 추천으로 행동이 바뀌는 유저 (CATE > 0, 랜덤 시 반응 낮음)
- **Sure Things**: 추천 없이도 반응하는 유저 (CATE ≈ 0, 랜덤 시 반응 높음)
- **Lost Causes**: 추천해도 반응 없는 유저 (CATE ≈ 0, 랜덤 시 반응 낮음)
- **Sleeping Dogs**: 추천하면 오히려 역효과 (CATE < 0)

In [ ]:
# 유저 4분류
combined['mu0'] = mu0  # 랜덤 시 예상 CTR
combined['mu1'] = mu1  # 추천 시 예상 CTR

cate_median = combined['cate'].median()
mu0_median = combined['mu0'].median()

def classify_user(row):
    if row['cate'] > cate_median and row['mu0'] <= mu0_median:
        return 'Persuadables'
    elif row['cate'] <= cate_median and row['mu0'] > mu0_median:
        return 'Sure Things'
    elif row['cate'] <= cate_median and row['mu0'] <= mu0_median:
        return 'Lost Causes'
    else:  # cate > median, mu0 > median
        return 'Sleeping Dogs'

combined['user_type'] = combined.apply(classify_user, axis=1)

# 분류 결과
type_counts = combined['user_type'].value_counts()
print("=== 유저 4분류 결과 ===\n")
for utype, count in type_counts.items():
    pct = count / len(combined) * 100
    print(f"  {utype:15s}: {count:,} ({pct:.1f}%)")

# 각 유형별 특성
type_summary = combined.groupby('user_type').agg(
    avg_cate=('cate', 'mean'),
    avg_mu0=('mu0', 'mean'),
    avg_mu1=('mu1', 'mean'),
    avg_active=('active_level', 'mean'),
    avg_fans=('fans_user_num', 'mean'),
    count=('cate', 'count')
).round(4)
print("\n=== 유형별 특성 ===")
type_summary

In [ ]:
# 유저 4분류 시각화 — Scatter Plot
fig, ax = plt.subplots(figsize=(10, 8))

colors_map = {'Persuadables': '#2ecc71', 'Sure Things': '#3498db', 
              'Lost Causes': '#95a5a6', 'Sleeping Dogs': '#e74c3c'}

for utype, color in colors_map.items():
    mask = combined['user_type'] == utype
    ax.scatter(combined[mask]['mu0'], combined[mask]['cate'], 
              alpha=0.3, s=10, color=color, label=f'{utype} ({mask.sum():,})')

ax.axhline(y=cate_median, color='black', linestyle='--', alpha=0.3)
ax.axvline(x=mu0_median, color='black', linestyle='--', alpha=0.3)

ax.set_xlabel('μ₀ (Expected CTR without Recommendation)', fontsize=12)
ax.set_ylabel('CATE (Recommendation Effect)', fontsize=12)
ax.set_title('User Segmentation — Uplift Quadrant Plot', fontsize=14)
ax.legend(markerscale=5, fontsize=11)

# 사분면 레이블
ax.text(0.02, combined['cate'].max()*0.9, 'Persuadables\n(추천이 필요한 유저)', fontsize=10, color='#2ecc71', fontweight='bold')
ax.text(mu0_median*1.2, combined['cate'].max()*0.9, 'Sleeping Dogs\n(추천하면 역효과)', fontsize=10, color='#e74c3c', fontweight='bold')
ax.text(0.02, combined['cate'].min()*0.5, 'Lost Causes\n(추천해도 무반응)', fontsize=10, color='#95a5a6', fontweight='bold')
ax.text(mu0_median*1.2, combined['cate'].min()*0.5, 'Sure Things\n(추천 없이도 반응)', fontsize=10, color='#3498db', fontweight='bold')

plt.tight_layout()
plt.savefig('../notebooks/figures/03_uplift_quadrant.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Feature Importance — 추천 효과를 결정하는 유저 특성

In [ ]:
# CATE를 타겟으로 한 feature importance
cate_model = GradientBoostingRegressor(n_estimators=100, max_depth=4, random_state=42)
cate_model.fit(X_all, combined['cate'].values)

importances = pd.Series(cate_model.feature_importances_, index=covariates).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(10, 6))
importances.plot(kind='barh', ax=ax, color='#9b59b6', alpha=0.8)
ax.set_title('Feature Importance for CATE Prediction', fontsize=14)
ax.set_xlabel('Importance')
plt.tight_layout()
plt.savefig('../notebooks/figures/03_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n추천 효과를 가장 크게 좌우하는 특성:")
for feat, imp in importances.tail(3).items():
    print(f"  {feat}: {imp:.4f}")

---
## 6. 인과추론 결론

### 주요 발견
1. **PSM 결과**: 공변량 균형을 맞춘 후에도 추천의 인과적 CTR 효과가 유의하게 존재
2. **CATE 분포**: 추천 효과는 유저마다 이질적 — 일부 유저에게 효과가 크고, 일부는 거의 없음
3. **Uplift 4분류**: Persuadables에게 추천 자원을 집중하면 효율 극대화 가능
4. **핵심 특성**: 활동도, 팔로워 수, 가입 기간 등이 추천 효과를 결정하는 주요 변수

### 비즈니스 제안
- **Persuadables** 유저를 타겟으로 추천 강화 → ROI 극대화
- **Sure Things** 유저는 추천 없이도 반응 → 추천 비용 절감 가능
- **Lost Causes** 유저에 대한 추천은 비효율 → 다른 마케팅 채널 고려
- **Sleeping Dogs** 유저에게는 추천 자제 → 이탈 방지

### Next Steps
- `04_ope.ipynb`: KuaiRec 완전관측 행렬로 Off-Policy Evaluation
- `05_filter_bubble.ipynb`: 추천의 콘텐츠 다양성 영향 분석